In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "AVAXUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-09-01 00:00:00+00:00,23.40,23.40,23.34,23.36,1949.78,2025-09-01 00:00:59.999999+00:00,45551.6510,245,990.53,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000000,0.000000e+00,0.000000,NaN,NaN
1,2025-09-01 00:01:00+00:00,23.37,23.38,23.36,23.38,2749.32,2025-09-01 00:01:59.999999+00:00,64252.5574,117,1277.50,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000449,2.492877e-04,0.000199,NaN,NaN
2,2025-09-01 00:02:00+00:00,23.37,23.37,23.34,23.35,2469.23,2025-09-01 00:02:59.999999+00:00,57645.9984,202,540.23,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.000359,1.770022e-07,-0.000359,NaN,NaN
3,2025-09-01 00:03:00+00:00,23.35,23.36,23.33,23.34,1112.24,2025-09-01 00:03:59.999999+00:00,25958.7190,136,289.20,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.001078,-3.650454e-04,-0.000713,NaN,NaN
4,2025-09-01 00:04:00+00:00,23.34,23.34,23.27,23.28,15199.03,2025-09-01 00:04:59.999999+00:00,354054.1022,540,5331.10,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.003834,-1.396899e-03,-0.002437,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 263,876
[info] optuna train rows: 168,880
[info] valid rows:        42,220
[info] test rows:         52,776


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-20 04:59:06,336] A new study created in memory with name: no-name-57095d77-022f-4e16-8a82-a6b677acecd4


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:04<?, ?it/s]

Best trial: 0. Best value: 0.0669626:   0%|          | 0/50 [00:04<?, ?it/s]

Best trial: 0. Best value: 0.0669626:   2%|▏         | 1/50 [00:04<03:26,  4.22s/it]

...

Best trial: 28. Best value: 0.0805881:  98%|█████████▊| 49/50 [03:05<00:02,  2.64s/it]

Best trial: 28. Best value: 0.0805881:  98%|█████████▊| 49/50 [03:05<00:02,  2.64s/it]

Best trial: 28. Best value: 0.0805881: 100%|██████████| 50/50 [03:05<00:00,  6.63s/it]

Best trial: 28. Best value: 0.0805881: 100%|██████████| 50/50 [03:05<00:00,  3.71s/it]

[I 2026-03-20 05:02:11,978] Trial 49 finished with value: 0.07529164568766511 and parameters: {'n_estimators': 2000, 'max_depth': 12, 'learning_rate': 0.0013824476420208301, 'subsample': 0.7635056960892791, 'colsample_bytree': 0.8425014434062916, 'min_child_weight': 9, 'reg_alpha': 0.019990642607949284, 'reg_lambda': 4.713443227807427e-06}. Best is trial 28 with value: 0.08058808339764677.

[optuna] best trial
value: 0.080588
params:
  n_estimators: 400
  max_depth: 11
  learning_rate: 0.0074713837544292915
  subsample: 0.848723740736333
  colsample_bytree: 0.5612259620127913
  min_child_weight: 9
  reg_alpha: 0.132843253334049
  reg_lambda: 1.2823543304659255e-07


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final xgb...


[training] done in 4.94s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:      0.798947
Test IC:       -0.016092
Train Rank IC: 0.298158
Test Rank IC:  0.060014
Train RMSE:    0.003822
Test RMSE:     0.003217


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
volume_mom_5        0.119665
imbalance_15        0.090763
trend_strength      0.088732
num_trades_mom_5    0.088199
dist_ma_15_z        0.072651
range_ratio         0.055390
is_trending         0.048959
mr_x_vol            0.039228
trend_x_imb         0.036855
trades_z            0.036836
mom_x_imb           0.035724
dom_sin             0.031953
volume_z            0.030408
dist_ma_5           0.028414
hour_cos            0.022145
imbalance           0.018173
vol_ratio_5_30      0.017185
mom_60              0.012986
mom_3               0.011808
vol_15              0.011023
vol_regime_ratio    0.010911
atr_norm            0.010694
imbalance_5         0.009889
vol_30              0.009460
mom_5               0.007047
range_15            0.007029
vol_5               0.006790
hour_sin            0.005358
mom_30              0.005326
dow_cos             0.004199
dist_ma_15          0.003271
mom_15              0.003212
dist_ma_30          0.003118
macd_hist  

In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/AVAXUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/AVAXUSDT__h5_model.joblib
[saved] features -> models/xgb/AVAXUSDT__h5_feature_cols.json
[saved] feature importance -> models/xgb/AVAXUSDT__h5_feature_importance.csv
[saved] metadata -> models/xgb/AVAXUSDT__h5_meta.json
